# VSCode Jupyter 一键自检 Notebook

**使用方式**：
1. 打开本文件后，右上角点击 `Select Kernel`，选择装了 `ipykernel` 的 Python 环境
2. 每个单元格左侧点 ▶️ 或按 `Shift+Enter` 逐格运行
3. 想一键全跑：顶部工具栏 `Run All`

## 1. 环境自检

In [1]:
import load_dotenv
from typing_extensions import override
from urllib3.contrib.emscripten import response

print("Python 版本 :hello")
print('Hello')

ModuleNotFoundError: No module named 'js'

In [ ]:
from dotenv import load_dotenv
from deepagents import create_deep_agent
from rich import print

load_dotenv(override=True)

# def get_weather(city: str) -> str:
#     """Get weather for a given city."""
#     return f"It's always sunny in {city}!"
#
#
# agent = create_deep_agent(
#     model="openai:gpt-5.5",
#     tools=[get_weather],
#     system_prompt="You are a helpful assistant",
# )
#
# # Run the agent
# response = agent.invoke(
#     {"messages": [{"role": "user", "content": "what is the weather in sf"}]}
# )

# print(response)

# Step 3: Create a search tool

In [ ]:
import os
from typing import Literal

from tavily import TavilyClient
from deepagents import create_deep_agent

tavily_client = TavilyClient(api_key=os.environ["TAVILY_API_KEY"])

load_dotenv(override=True)

def internet_search(
    query: str,
    max_results: int = 5,
    topic: Literal["general", "news", "finance"] = "general",
    include_raw_content: bool = False,
):
    """Run a web search"""
    return tavily_client.search(
        query,
        max_results=max_results,
        include_raw_content=include_raw_content,
        topic=topic,
    )

# Step 4: Create a deep agent

In [ ]:
# System prompt to steer the agent to be an expert researcher
research_instructions = """You are an expert researcher. Your job is to conduct thorough research and then write a polished report.

You have access to an internet search tool as your primary means of gathering information.

## `internet_search`

Use this to run an internet search for a given query. You can specify the max number of results to return, the topic, and whether raw content should be included.
"""

agent = create_deep_agent(
    model="openai:gpt-5.5",
    tools=[internet_search],
    system_prompt=research_instructions,
)

# LangSmith

In [ ]:
from rich import print

result = agent.invoke({"messages": [{"role": "user", "content": "What is langgraph?"}]})

print(result)

# Print the agent's response
print(result["messages"][-1].content)

# Custom middleware

In [ ]:
from langchain.agents.middleware import wrap_tool_call
from langchain.tools import tool
from deepagents import create_deep_agent
from rich import print


@tool
def get_weather(city: str) -> str:
    """Get the weather in a city."""
    return f"The weather in {city} is sunny."


call_count = [0]  # Use list to allow modification in nested function


@wrap_tool_call
def log_tool_calls(request, handler):
    """Intercept and log every tool call - demonstrates cross-cutting concern."""
    call_count[0] += 1
    tool_name = request.name if hasattr(request, "name") else str(request)

    print(f"[Middleware] Tool call #{call_count[0]}: {tool_name}")
    print(f"[Middleware] Arguments: {request.args if hasattr(request, 'args') else 'N/A'}")

    # Execute the tool call
    result = handler(request)

    # Log the result
    print(f"[Middleware] Tool call #{call_count[0]} completed")

    return result


agent = create_deep_agent(
    model="openai:gpt-5.5",
    tools=[get_weather],
    middleware=[log_tool_calls],
)

result = agent.invoke({"messages": [{"role": "user", "content": "北京天气怎么样,并打印日志"}]})

print(result)

# Override a default middleware instance

In [ ]:
from deepagents import create_deep_agent
from deepagents.backends import StateBackend
from deepagents.middleware import SummarizationMiddleware
from rich import print

backend = StateBackend()
model = "openai:gpt-5.5"

custom_summarization = SummarizationMiddleware(
    model=model,
    backend=backend,
    summary_prompt="Your custom summary prompt here.",
)

agent = create_deep_agent(
    model=model,
    middleware=[custom_summarization],  # replaces the default SummarizationMiddleware
)

from rich import print
result = agent.invoke({"messages": [{"role": "user", "content": "Hello"}]})
print(result)

# Interpreters

In [ ]:
from deepagents import create_deep_agent
from langchain_quickjs import CodeInterpreterMiddleware

agent = create_deep_agent(
    model="openai:gpt-5.5",
    middleware=[CodeInterpreterMiddleware()],
)


from rich import print
result = agent.invoke({"messages": [{"role": "user", "content": "Hello"}]})
print(result)

# Subagents

In [ ]:
import os
from typing import Literal

from deepagents import create_deep_agent
from tavily import TavilyClient

tavily_client = TavilyClient(api_key=os.environ["TAVILY_API_KEY"])


def internet_search(
    query: str,
    max_results: int = 5,
    topic: Literal["general", "news", "finance"] = "general",
    include_raw_content: bool = False,
):
    """Run a web search"""
    return tavily_client.search(
        query,
        max_results=max_results,
        include_raw_content=include_raw_content,
        topic=topic,
    )


research_subagent = {
    "name": "research-agent",
    "description": "Used to research more in depth questions",
    "system_prompt": "You are a great researcher",
    "tools": [internet_search],
    "model": "openai:gpt-5.5",  # Optional override, defaults to main agent model
}
subagents = [research_subagent]

agent = create_deep_agent(
    model="openai:gpt-5.5",
    subagents=subagents,
)

from rich import print
result = agent.invoke({"messages": [{"role": "user", "content": "Hello"}]})
print(result)

# Backends

In [ ]:
from deepagents import create_deep_agent
from deepagents.backends import StateBackend

# By default, we provide a StateBackend
agent = create_deep_agent(model="openai:gpt-5.5")

# Under the hood, it looks like
agent2 = create_deep_agent(
    model="openai:gpt-5.5",
    backend=StateBackend(),
)

result = agent.invoke({"messages": [{"role": "user", "content": "Hello"}]})
from rich import print
print(result)

In [2]:
import os

from deepagents import create_deep_agent
from langchain_runloop import RunloopSandbox
from runloop_api_client import RunloopSDK

from dotenv import load_dotenv
load_dotenv(override=True)

client = RunloopSDK(bearer_token=os.environ["RUNLOOP_API_KEY"])

devbox = client.devbox.create()
backend = RunloopSandbox(devbox=devbox)

agent = create_deep_agent(
    model="openai:gpt-5.5",
    system_prompt="You are a Python coding assistant with sandbox access.",
    backend=backend,
)

try:
    result = agent.invoke(
        {
            "messages": [
                {
                    "role": "user",
                    "content": "Create a small Python package and run pytest",
                }
            ]
        }
    )
finally:
    devbox.shutdown()

result = agent.invoke({"messages": [{"role": "user", "content": "Hello"}]})
from rich import print
print(result)

KeyboardInterrupt: 

# Human-in-the-loop

In [7]:
from langchain.tools import tool
from deepagents import create_deep_agent
from langgraph.checkpoint.memory import MemorySaver


@tool
def remove_file(path: str) -> str:
    """Delete a file from the filesystem."""
    return f"Deleted {path}"


@tool
def fetch_file(path: str) -> str:
    """Read a file from the filesystem."""
    return f"Contents of {path}"


@tool
def notify_email(to: str, subject: str, body: str) -> str:
    """Send an email."""
    return f"Sent email to {to}"


# Checkpointer is REQUIRED for human-in-the-loop
checkpointer = MemorySaver()

agent = create_deep_agent(
    model="openai:gpt-5.5",
    tools=[remove_file, fetch_file, notify_email],
    interrupt_on={
        "remove_file": True,  # Default: approve, edit, reject, respond
        "fetch_file": False,  # No interrupts needed
        "notify_email": {"allowed_decisions": ["approve", "reject"]},  # No editing
    },
    checkpointer=checkpointer,  # Required!
)

session_config = {
    "configurable": {
        "thread_id": "chat_session_001"  # 自定义任意唯一字符串
    }
}

result = agent.invoke(
    {"messages": [{"role": "user", "content": "将hello写入本地文件"}]},
    config=session_config
)
from rich import print
print(result)

{
    'messages': [
        HumanMessage(
            content='将hello写入本地文件',
            additional_kwargs={},
            response_metadata={},
            id='deb6e58b-d619-43f9-9442-8bc603c2dd68'
        ),
        AIMessage(
            content=[
                {
                    'id': 'rs_02178379182826200000000000000000000ffffac15d14098dbee',
                    'summary': [
                        {
                            'text': 
'\n用户让我把hello写入本地文件，首先我需要确定用什么工具，这里是write_file对吧？不过等下，用户没说具体文件名和路径
？不对，或者是不是默认可以用一个常见的路径？不对，等下，工具要求file_path是绝对路径，那我可以选比如/tmp/hello.txt？
或者是不是应该问用户？不对不对，用户说本地文件，可能默认的话先写个比如/hello.txt？不对不对，等下，有没有更合理的？
哦不对，或者是不是用户希望指定？不对，不对，再看用户的请求，就只是说将hello写入本地文件，没说文件名，那我是不是应该
先问用户文件名和路径？不对不对，等下，会不会默认可以用一个测试文件名？比如/hello.txt？对，就先写这个吧，内容是hello
，对吧？那调用write_file工具，file_path是/hello.txt，content是"hello"。对吧？\n\n哦等下，不对，要不要先检查有没有这
个文件？不用吧，write_file是创建新文件，如果存在的话？不对，工具描述说write_file是创建新文件，prefer edit existing 
over create 
new。但用户就是要写入，不管有没有？或者是不是先ls看看？不对，用户的需求很简单，就是写hello到本地文件，那直接写就行
，就用/hello.txt吧，内容hello。',
                            'type': 'summary_text'
                        }
                    ],
                    'type': 'reasoning',
                    'status': 'completed'
                },
                {
                    'arguments': '{"file_path": "/hello.txt", "content": "hello"}',
                    'call_id': 'call_16n157cv27boseldqghsv3i6',
                    'name': 'write_file',
                    'type': 'function_call',
                    'id': 'fc_02178379183250900000000000000000000ffffac15d140ae358b',
                    'status': 'completed'
                }
            ],
            additional_kwargs={},
            response_metadata={
                'id': 'resp_021783791824036c5b3f9d60cef2f4f5261bc47633cdc22687ad8',
                'created_at': 1783791828.0,
                'model': 'doubao-seed-2.0-pro',
                'object': 'response',
                'service_tier': 'default',
                'status': 'completed',
                'model_provider': 'openai',
                'model_name': 'doubao-seed-2.0-pro'
            },
            id='resp_021783791824036c5b3f9d60cef2f4f5261bc47633cdc22687ad8',
            tool_calls=[
                {
                    'name': 'write_file',
                    'args': {'file_path': '/hello.txt', 'content': 'hello'},
                    'id': 'call_16n157cv27boseldqghsv3i6',
                    'type': 'tool_call'
                }
            ],
            invalid_tool_calls=[],
            usage_metadata={
                'input_tokens': 6487,
                'output_tokens': 349,
                'total_tokens': 6836,
                'input_token_details': {'cache_read': 5944},
                'output_token_details': {'reasoning': 300}
            }
        ),
        ToolMessage(
            content='Updated file /hello.txt',
            name='write_file',
            id='996bde94-9525-4ae3-9d6e-1c51dd3ee449',
            tool_call_id='call_16n157cv27boseldqghsv3i6'
        ),
        AIMessage(
            content=[
                {
                    'id': 'rs_02178379183624100000000000000000000ffffac1506218f2428',
                    'summary': [
                        {
                            'text': 
'\n用户让我把hello写入本地文件，刚才调用write_file工具已经成功更新了/hello.txt，现在需要确认任务完成了对吧？首先检
查一下，是不是写入成功了？可以调用read_file看看内容对不对？哦对，最好验证一下，避免出错。那我调用read_file读取/hell
o.txt的内容看看是不是hello。',
                            'type': 'summary_text'
                        }
                    ],
                    'type': 'reasoning',
                    'status': 'completed'
                },
                {
                    'arguments': '{"file_path": "/hello.txt"}',
                    'call_id': 'call_mggzgp0c8ac27pwbtlcesma6',
                    'name': 'read_file',
                    'type': 'function_call',
                    'id': 'fc_02178379183722200000000000000000000ffffac150621683d85',
                    'status'